# 人脸追踪 face follow play

### 导入头文件 Import head file

In [1]:
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from face_follow import Face_Follow
from dofbot_utils.fps import FPS
from dofbot_utils.robot_controller import Robot_Controller
from dofbot_utils.dofbot_config import *

### 初始化机械臂位置 Initialize DOFBOT position

In [2]:
robot = Robot_Controller()
robot.move_init_pose()
fps = FPS()

### 创建实例,初始化参数 Create the instance and initialize the parameters

In [3]:
follow = Face_Follow()
model = 'General'

haar_path: /home/yahboom/dofbot_pro/dofbot_face_follow/scripts/haarcascade_frontalface_default.xml


### 创建控件 Creating widget

In [4]:
button_layout = widgets.Layout(width='250px', height='50px', align_self='center')
output = widgets.Output()
# 退出控件 exit button
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件 Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
# 空间布局 spatial distribution
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset']

### 模式切换 Switch mode

In [5]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
#     with output: print(model)
exit_button.on_click(exit_button_Callback)

### 主程序 Main process

In [6]:
def camera():
    global model
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0, cv.CAP_V4L2)
    capture.set(3, 640)
    capture.set(4, 480)
    capture.set(5, 30) 
    while capture.isOpened():
        try:
            _, img = capture.read()
            fps.update_fps()
            img, pos = follow.follow_function(img)
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

### 启动  Start

In [7]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()